# Daniel Peña Fonseca - 20031020-T330
Use of LLMs (Claude Sonet 4.6) for this assignment:

- Exercise 5.1: for loading, it was helpful for noticing that a change of line should be characterized as $<eos>$. Initially, a certain code was implemented, where windows were used for prediction, and an easier structure for the model was used. After multiple trials with different parameters, using Claude was hepful for setting BPTT, which noticeably decreased the perplexity. For the function compute_perplexity, it was also particularly useful. For the run, it was helpful for getting to know what functions does 'model.' have, like 'model.train_on_batches', 'model.test_on_batches', or 'model.get_weights' when the validation loss decreased, so we can keep the weights of the best model. Also for the function reset_model_states.

- Exercise 5.2: In this case, it was used to make the code faster, not particularly helpful in one matter. Although, it showed me that, when computing a multiplication of small values, like probabilities, it is better (numerically) to take the sum of logarithms and then take the exp, which I did not know.

- Exercise 5.3: Not particularly helpful for this exercise.

- Exercise 5.4: Although it was nos specially useful in this case, it helped me with the handling of the indices, particularly when eliminating the element that falls out of the window.

- Exercise 5.5: Since I had no experience with sentence generation, it was really helpful for this exercise; it was used for all the functions.

# Exercise 5

In [ ]:
%pip install -q tensorflow numpy

In [81]:
import tensorflow as tf
import os
import math
import numpy as np
from tensorflow.keras.layers import TimeDistributed
from collections import Counter
import random
import json
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint

# Exercise 5.1

In this exercise we just have to download the different data sets from the link given in the statement of the exercise. Then we define a function for loading the data, where we replace the newline \n for $<eos>$, since we want to consider each training set as a 'long sentence. Finally, we load the data using the function.

In [82]:
# Download
url = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/"
files = ['ptb.train.txt', 'ptb.valid.txt', 'ptb.test.txt']

for file in files:
    # tf.keras.utils.get_file(file neme, original url of the file)
    tf.keras.utils.get_file(file, url+file)

def load_data(filename):
    # .keras/datasets is where tf.keras.utils.get_file stores the files.
    path = os.path.join(os.path.expanduser('~'), '.keras/datasets', filename)
    with open(path, 'r', encoding='utf-8') as f:
        # Checking the files, we can see that lowercase is used, and <unk> too, but no <eos> can be found.
        # Therefore, we replace each newline with eos.
        return f.read().replace('\n', '<eos>')
        
# Load
train_text = load_data('ptb.train.txt')
valid_text = load_data('ptb.valid.txt')
test_text = load_data('ptb.test.txt')

In this exercise, we will apply an LSTM for a language model. Procedure:
- We first tokenize the training set, since we need to separate the 'long sentence' into tokens for the LSTM to be able to work. Fitting the tokenizer on the training text (tokenizer.fit_on_texts), we extract the vocabulary that we will be working with. However, another step is required to turn the text into a format that the LSTM can 'understand', which is turning the text into ids. 
- Next, we define how windows will be created. Since LSTMs use the previous window_size values (ids) to predict the next one, we need to separate the data into windows that will work as inputs for the LSTM. Instead of using a stateless LSTM, where the information from past windows is not considered, we ended up using back propagation through time (bptt), trying to capture more information. Setting a bppt length, we define how far back the model can learn dependencies within one update.  
- Finally, we create the model, specifying the number of units for the LSTM and the number of layers.

First, we search for the set (bptt size, lstm size, number of layers) for which the best results are obtained; i.e. the set for which the minimum validation loss among all its epochs is the minimum when compared to this value from other sets of (bptt size, lstm size, number of layers). This way, we will get the optimal value of the window size, the LSTM size, and the number of layers for the model; along with the best model (i.e. the model at the epoch at which the lowest validation loss appeared).

After studying plenty of parameter combinations, the ones presented in the following cell were the chosen ones. Using early stopping, we search for the model that yields the lowest validation loss. 


In [88]:

print("GPUs available:", tf.config.list_physical_devices('GPU')) # try using gpu
print(tf.__version__)

tokenizer = Tokenizer() # defining our tokenizer
tokenizer.fit_on_texts([train_text]) # we create the vocabulary using our training data
vocab_size = len(tokenizer.word_index) + 1 # the +1 added is used because index 0 will indicate unknown tokens or padding

def text_to_ids(text):
    return tokenizer.texts_to_sequences([text])[0] # we turn a string into ids so that our model can understand
# without the [0], we have a lists of sublists, where each sublist corresponds to a text. But here we have only one long text, so we will have
# a list of one sublist. Since we want to work with the sublist, we use [0]

# we apply the transformation for all datasets
train_ids = np.array(text_to_ids(train_text))
valid_ids = np.array(text_to_ids(valid_text))
test_ids  = np.array(text_to_ids(test_text))


def reset_model_states(model):
    for layer in model.layers:
        if hasattr(layer, 'reset_states'): # We need to reset the states of the lstm
            layer.reset_states()

def make_bptt_dataset(ids, batch_size, bptt_len):
    n_batches = (len(ids) - 1) // (batch_size * bptt_len)
    ids = ids[:n_batches * batch_size * bptt_len + 1]
    x_data = ids[:-1].reshape(batch_size, -1) # as input we give all the tokens except the last one, and they are turned into batch_size, sequence length, so we structure them properly. Here we get lists of batch_size.
    y_data = ids[1:].reshape(batch_size, -1) # here we shift, since we want to predict next word
    sequences_x, sequences_y = [], []
    for i in range(0, x_data.shape[1], bptt_len):
        sequences_x.append(x_data[:, i:i+bptt_len]) # now we slice each list of batch size length into chunks
        sequences_y.append(y_data[:, i:i+bptt_len]) # targets for that chunk
    return sequences_x, sequences_y

def build_model(num_layers, lstm_size, batch_size, bptt_len, vocab_size):
    inputs = tf.keras.Input(batch_shape=(batch_size, bptt_len)) # when we use stateful LSTM, we need this change
    x = Embedding(input_dim=vocab_size, output_dim=650)(inputs) # really important. instead of using normal integeers ids, it will use vectors of length 650, to make sure that words that are similar have similar vectors. We make sure about this during the training. Here each word will be represented as a vector of 650 numbers
    x = Dropout(0.3)(x)
    for layer_idx in range(num_layers):
        x = LSTM(lstm_size, dropout=0.6, stateful=True, return_sequences=True)(x) # return_sequences makes sure it is stateful
        x = Dropout(0.6)(x)
    outputs = TimeDistributed(Dense(vocab_size, activation='softmax'))(x) # Dense(...) outputs a probability for each token in the vocabulary, given what it has seen. With time distributed, we apply this independently to each of the bppt_len vectors
    
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def compute_perplexity(model, ids, batch_size, bptt_len):
    sequences_x, sequences_y = make_bptt_dataset(ids, batch_size, bptt_len)
    reset_model_states(model) # we reset before starting
    total_log_prob = 0.0 # instead of multiplying probabilities, we sum the logarithms and then we take the exp
    n = 0
    for x_batch, y_batch in zip(sequences_x, sequences_y):
        x_batch = np.array(x_batch)
        y_batch = np.array(y_batch)
        if x_batch.shape[0] != batch_size:
            continue
        probs = model.predict(x_batch, verbose=0) # we generate probabilities. shape (batch_size, bptt_len, vocab_size)
        # we loop over each time step
        for t in range(y_batch.shape[1]):
            correct_probs = probs[np.arange(batch_size), t, y_batch[:, t]] # the perplexity studies how confused is the model that the next token is the one it is. So the probability that we use is the probability that the model assigns to the correct token
            total_log_prob += np.sum(np.log(correct_probs + 1e-10)) # we sum. The +1e-10 is just in case the model assigned a probability 0, since then the log would be infinity
            n += batch_size
    return math.exp(-total_log_prob / n)

# Parameters
batch_size = 64
bptt_len   = 70 # size of each chunk
num_layers_options = [2]
lstm_size_options  = [650]
number_of_epochs   = 250

# Prepare data
train_sequences_x, train_sequences_y = make_bptt_dataset(train_ids, batch_size, bptt_len)
valid_sequences_x, valid_sequences_y = make_bptt_dataset(valid_ids, batch_size, bptt_len)
x_train = np.array(train_sequences_x)
y_train = np.array(train_sequences_y)
x_valid = np.array(valid_sequences_x)
y_valid = np.array(valid_sequences_y)

num_layers = num_layers_options[0]
lstm_size = lstm_size_options[0]


seed = 88
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
2.19.0


In [89]:
print(f"\nTraining: layers={num_layers}, lstm_size={lstm_size}, bptt={bptt_len}")

model = build_model(num_layers, lstm_size, batch_size, bptt_len, vocab_size)
#checkpoint_path = f"/kaggle/working/best_model_layers{num_layers}_lstm{lstm_size}_2.keras"

best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0 # for early stopping
patience = 10 # for early stopping
lr_patience_counter = 0 # to change the learning rate
lr_patience = 4 # to change the learning rate
best_weights = None

for epoch in range(number_of_epochs):
    reset_model_states(model) # we clear the memory
    train_losses = []
    # we loop through every bptt chunk
    for x_chunk, y_chunk in zip(x_train, y_train):
        loss = model.train_on_batch(x_chunk, y_chunk) # training
        train_losses.append(loss[0]) # loss is [loss_value, accuracy], so we just keep the loss of this chunk

    reset_model_states(model)
    val_losses = []
    for x_chunk, y_chunk in zip(x_valid, y_valid):
        loss = model.test_on_batch(x_chunk, y_chunk) # only forward pass, we do not train now, as we did before with train_on_batch
        val_losses.append(loss[0])

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    print(f"Epoch {epoch+1} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")
    
    # now we save the best model:

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        #model.save(checkpoint_path)  # ← save weights only
        best_weights = model.get_weights()
        print(f"  >> Saved best model at epoch {best_epoch}")
        #print(f"  >> File exists: {os.path.exists(checkpoint_path)}")  # ← add this
        #print(f"  >> File size: {os.path.getsize(checkpoint_path)} bytes")  # ← and this
        patience_counter = 0
        lr_patience_counter = 0
    else:
        patience_counter += 1
        lr_patience_counter += 1

    if lr_patience_counter >= lr_patience:
        old_lr = float(model.optimizer.learning_rate)
        new_lr = max(old_lr * 0.5, 1e-6)
        model.optimizer.learning_rate = new_lr
        print(f"  >> Reduced LR to {new_lr:.6f}")
        lr_patience_counter = 0

    if patience_counter >= patience:
        print(f"  >> Early stopping at epoch {epoch+1}")
        break

print(f"\n>>> Best val_loss={best_val_loss:.4f} at epoch {best_epoch}")




Training: layers=2, lstm_size=650, bptt=70
Epoch 1 | train_loss=7.0653 | val_loss=6.7505
  >> Saved best model at epoch 1
Epoch 2 | train_loss=6.6300 | val_loss=6.5079
  >> Saved best model at epoch 2
Epoch 3 | train_loss=6.4038 | val_loss=6.3099
  >> Saved best model at epoch 3
Epoch 4 | train_loss=6.2303 | val_loss=6.1580
  >> Saved best model at epoch 4
Epoch 5 | train_loss=6.0963 | val_loss=6.0397
  >> Saved best model at epoch 5
Epoch 6 | train_loss=5.9903 | val_loss=5.9447
  >> Saved best model at epoch 6
Epoch 7 | train_loss=5.9041 | val_loss=5.8663
  >> Saved best model at epoch 7
Epoch 8 | train_loss=5.8317 | val_loss=5.7993
  >> Saved best model at epoch 8
Epoch 9 | train_loss=5.7695 | val_loss=5.7413
  >> Saved best model at epoch 9
Epoch 10 | train_loss=5.7151 | val_loss=5.6904
  >> Saved best model at epoch 10
Epoch 11 | train_loss=5.6671 | val_loss=5.6450
  >> Saved best model at epoch 11
Epoch 12 | train_loss=5.6239 | val_loss=5.6038
  >> Saved best model at epoch 12
Ep

In [90]:
# Using the best model (lowest validation loss), we compute the validation perplexity
model.set_weights(best_weights)
reset_model_states(model)
val_losses = []
for x_chunk, y_chunk in zip(x_valid, y_valid):
    loss = model.test_on_batch(x_chunk, y_chunk)
    val_losses.append(loss[0])
print(f"Val loss (current model in memory): {np.mean(val_losses):.4f}")
print(f"Val Perplexity: {math.exp(np.mean(val_losses)):.2f}")

Val loss (current model in memory): 4.3027
Val Perplexity: 73.90


In [91]:
# since we did separate in windows the test set before, we do it now. Here, we compute the perplexity of the test set using the best model obtained during the training
reset_model_states(model)
test_seq_x, test_seq_y = make_bptt_dataset(test_ids, 64, 70)
reset_model_states(model)
test_losses = []
for x_chunk, y_chunk in zip(test_seq_x, test_seq_y):
    loss = model.test_on_batch(x_chunk, y_chunk)
    test_losses.append(loss[0])
print(f"Test loss: {np.mean(test_losses):.4f}")
print(f"Test Perplexity: {math.exp(np.mean(test_losses)):.2f}")

Test loss: 4.3029
Test Perplexity: 73.91


# Exercise 5.2

In this case, we will use the interpretable model of n-grams for language prediction. Here, we want to predict the next token using the n-grams statistics learned from the training data; i.e. the probability that the next token is $w_i$, given that the previous token was $w_{i-1}$, and the one before was $w_{i-2}$ is:
$$
P_{JM}(w_i | w_{i-2}, w_{i-1}) = c_1 P(w_i | w_{i-2}, w_{i-1}) + c_2 P(w_i | w_{i-1}) + (1-c_1-c_2)P(w_{i})
$$
That is, using the Jelinek-Mercer smoothing. Therefore, the probability that the token w_i comes next will be given by the probability of $w_i$ being there, given that the bigram $(w_{i-2}, w_{i-1})$ is present; by the probability of $w_i$ being there, given that the unigram $w_{i}$ is present; and by the probability of the unigram alone. The coefficients $c_1$ and $c_2$ will simply determine how much importance is placed under each of these probabilities. To determine which is the optimal combination $(c_1, c_2)$, we will try different values.

Note:
$$
P(w_i | w_{i-2}, w_{i-1}) = \frac{P(w_{i-2}, w_{i-1}, w_i)}{P(w_{i-2}, w_{i-1})} = \frac{count(w_{i-2}, w_{i-1}, w_{i})}{count(w_{i-2}, w_{i-1})}
$$

$$
P(w_{i-1} | w_{i-2}) = \frac{P(w_{i-1}, w_i)}{P(w_{i-1})} = \frac{count(w_{i-1}, w_i)}{count(w_{i-1})}
$$

$$
P(w_{i}) = \frac{P(w_i)}{1} = \frac{count(w_i)}{\text{Number of unigrams (text length)}}
$$
I.e. in the first case, we want to measure the proportion of bigrams $(w_{i-2}, w_{i-1})$ that end up in the trigram $(w_{i-2}, w_{i-1}, w_{i})$; in the second case we want to measure the proportion of unigrams $w_{i-1}$ that end up in the bigram $(w_{i-1}, w_{i})$. Finally, we want to capture how common is the presence of $w_i$ in the training set.


For this matter, we follow this procedure:
- First, we tokenize the different data sets. Since we want to train the model using the training data, we extract all the unigrams, bigrams and trigrams from the data set, and we count how many times do each of them appear.
- Then, we create the functions for the probabilities described above, and a function to determine the perplexity of the model using:
$$
PP(\text{Text})= \left( \prod_{i}{P_{JM}(w_{i} | w_{i-2}, w_{i-1})}\right)
$$
- Finally, we study the validation perplexity obtained for each set $(c_1, c_2)$ studied, and keep the combination $(c_1, c_2)$ for which the best results are obtained.

In [92]:
def tokenize(text):
    return text.strip().split()

# Tokenize each data set
train_tokens = tokenize(train_text)
valid_tokens = tokenize(valid_text)
test_tokens  = tokenize(test_text)

# Total number of tokens
total_train_tokens = len(train_tokens)
total_val_tokens = len(valid_tokens)
total_test_tokens = len(test_tokens)

print(f"Train tokens: {total_train_tokens}")
print(f"Valid tokens: {total_val_tokens}")
print(f"Test  tokens: {total_test_tokens}")

# Count ngrams on training set
counts1 = Counter(train_tokens) # unigrams
counts2 = Counter(zip(train_tokens[:-1], train_tokens[1:])) # bigrams
counts3 = Counter(zip(train_tokens[:-2], train_tokens[1:-1], train_tokens[2:])) # trigrams

print('\nTraining set: ')
print(f"    Vocabulary size : {len(counts1)}")
print(f"    Unique bigrams  : {len(counts2)}")
print(f"    Unique trigrams : {len(counts3)}")

# Probability function for each ngram type
# When counting the amount of times that an ngram appears in the text, we want to obtain 0 in case it does no appear
def p_trigram(w1, w2, w3):
    "The conditioned probability P(w3 | w1, w2) = P(w1,w2,w3) / P(w1, w2) = count(w1,w2,w3) / count(w1,w2)"
    denom = counts2.get((w1, w2), 0) 
    if denom == 0:
        return 0.0
    return counts3.get((w1, w2, w3), 0) / denom

def p_bigram(w1, w2):
    "The conditioned probability P(w2 | w1) = P(w1,w2) / P(w1) = count(w1,w2) / count(w1)"
    denom = counts1.get(w1, 0)
    if denom == 0:
        return 0.0
    return counts2.get((w1, w2), 0) / denom

def p_unigram(w):
    "The probability P(w) is the number of times that the token w appears in the text divided by the total tokens in the text (training)"
    return counts1.get(w, 0) / total_train_tokens

# Perplexity
def compute_perplexity(tokens, c1, c2):
    c3 = 1.0 - c1 - c2
    log_prob_sum = 0.0
    n = 0

    # We start in the third word so we can compute w_{i-2}, w_{i-1}
    for i in range(2, len(tokens)):
        w1, w2, w3 = tokens[i-2], tokens[i-1], tokens[i]

        # Equation of Jelinek-Mercer smoothing
        p = (c1 * p_trigram(w1, w2, w3) +
             c2 * p_bigram(w2, w3) +
             c3 * p_unigram(w3))

        # When there are unknown words, we will have p=0, which will explode the logarithm.
        p = max(p, 1e-10)
        log_prob_sum += math.log(p) # numerically better than computing the multiplication of probabilities
        n += 1

    return math.exp(-log_prob_sum / n)

# Grid search
c1_options = np.arange(0, 1.0, 0.1)
c2_options = np.arange(0, 1.0, 0.1)

configs = [(c1, c2) for c1 in c1_options
                    for c2 in c2_options
                    if c1 + c2 < 1.0]

print(f"\nStudying {len(configs)} combinations of (c_1, c_2)\n")

grid_results = []

i=0
for c1, c2 in configs:
    i+=1
    if i%100==0:
        print(f'{i}/{len(configs)}')
    val_perplexity = compute_perplexity(valid_tokens, c1, c2)
    grid_results.append((c1, c2, val_perplexity))
    
# Best configuration (minimum validation perplexity)
best_c1, best_c2, best_val_pp = min(grid_results, key=lambda x: x[2])
test_perplexity = compute_perplexity(test_tokens, best_c1, best_c2)

print(f"\nBest config → c1 = {best_c1}, c2 = {best_c2}, c3 = (1-c1-c2) = {1-best_c1-best_c2:.1f} "
      f"| val_perplexity={best_val_pp:.2f}   |    Test perplexity → {test_perplexity:.2f}")


Train tokens: 929589
Valid tokens: 73760
Test  tokens: 82430

Training set: 
    Vocabulary size : 10000
    Unique bigrams  : 264989
    Unique trigrams : 615128

Studying 55 combinations of (c_1, c_2)


Best config → c1 = 0.2, c2 = 0.5, c3 = (1-c1-c2) = 0.3 | val_perplexity=200.88   |    Test perplexity → 185.13


# Exercise 5.3

In this exercise, a more sophisticated approach is followed. Now, instead of having fixed values of $c_1$ and $c_2$, we will make these values dependent on the number of instances. If the bigram appearing before the token that we want to predict has only appeared a few times, we want $c_1$ (multiplying $P(w_i | w_{i-2}, w_{i-1})$) to be small, so that these few instances of the bigram do not have a high impact on the prediction. However, if the bigram appearing before the token that we want to predict has appeared many times, we want $c_1$ to be big, so that these instances of the bigram have a high impact on the prediction, since the model has been well trained for these occurences. 
The same principle is applied to $c_2$ taking into account the instances of the unigram appearing before the token that we want to predict.

For this implementation, we manually select values of instances and values of $c_1$ and $c_2$ that will be associated to each number of instances. These values were changed multiple times until obtaining the values presented in functions c1_from_bins, c2_from_bins, which worked the best out of all the ones tried.

Finally, we compute the validation and test perplexity in the same way as we did before.

In [93]:


def c1_from_bins(bigram_count):
    if bigram_count == 0:
        return 0.0
    elif bigram_count < 2:
        return 0.1
    elif bigram_count < 7:
        return 0.15
    elif bigram_count < 10:
        return 0.2
    elif bigram_count < 30:
        return 0.25
    elif bigram_count < 50:
        return 0.3
    elif bigram_count < 100:
        return 0.32
    else:
        return 0.4
    


def c2_from_bins(unigram_count):
    if unigram_count ==0:
        return 0.0
    elif unigram_count < 2:
        return 0.1
    elif unigram_count < 5:
        return 0.15
    elif unigram_count < 10:
        return 0.25
    elif unigram_count < 20:
        return 0.3
    elif unigram_count < 35:
        return 0.35
    elif unigram_count < 100:
        return 0.4
    else:
        return 0.5

def compute_perplexity_bins(tokens):
    log_prob_sum = 0.0
    n = 0

    for i in range(2, len(tokens)):
        w1, w2, w3 = tokens[i-2], tokens[i-1], tokens[i]

        bigram_count  = counts2.get((w1, w2), 0)
        unigram_count = counts1.get(w2, 0)

        c1 = c1_from_bins(bigram_count)
        c2 = c2_from_bins(unigram_count)
        c3 = 1.0 - c1 - c2

        p = (c1 * p_trigram(w1, w2, w3) +
             c2 * p_bigram(w2, w3) +
             c3 * p_unigram(w3))

        p = max(p, 1e-10)
        log_prob_sum += math.log(p)
        n += 1

    return math.exp(-log_prob_sum / n)


val_pp  = compute_perplexity_bins(valid_tokens)
test_pp = compute_perplexity_bins(test_tokens)
print(f'Validation perplexity: {val_pp:.2f}')
print(f"\nTest perplexity: {test_pp:.2f}")

Validation perplexity: 183.72

Test perplexity: 171.03


# Exercise 5.4

In this exercise, apart from the Jelinek-Mercer smoothing, we want to take into account another feature; that is 'cache'. Taking into account the cache, we are considering that, if multiple tokens out of the last $m$, where 'Token_i', then 'Token_i' has more chances of appearing as the next word. This seems a reasonable considerations, since, in most cases, the context is really important. Taking into account the training set is crucial for determining the probability of the next token being a certain one, but only taking into account the data set under study, as we did with the cache, considers important aspects, otherwise ignored. For example, if the current paragraph speaks about money, and the token 'N' appears multiple times, we want to give more probability to it (sum a positive value to $P_{JM}('N', w_{i-2}, w_{i-1})$), given that the context is favoring the appearance of this token.

To compute $P_{cache}(w_i)$, we calculate:
$$
P_{cache}(w_i)=\frac{\text{Number of times } w_i \text{ appeared in the } m \text{ tokens preceding } w_i}{m}
$$

The implementation just requires taking into account that, for the first $m$ tokens, the window of preceding tokens should kept growing. For token $m+1$, the window is already of size $m$; so from token $m+1$ onwards, computing the window will just imply adding the last token, and deleting the one that falls out of the window.

When compared to predictions computed using only Jelinek-Mercer smoothing with parameters $c_1$ and $c_2$ dependent on the instances, this model gives noticeably better results; with perplexity dropping significantly.

In [94]:
import math
import json
from collections import Counter


# Probability JM
def p_jm(w1, w2, w3):
    bigram_count  = counts2.get((w1, w2), 0)
    unigram_count = counts1.get(w2, 0)
    c1 = c1_from_bins(bigram_count)
    c2 = c2_from_bins(unigram_count)
    c3 = 1.0 - c1 - c2
    return (c1 * p_trigram(w1, w2, w3) +
            c2 * p_bigram(w2, w3) +
            c3 * p_unigram(w3))

# Perplexity
def compute_perplexity_cache(tokens, lam, m):
    """
    At each position i (starting from 2), the cache contains
    the previous min(i, m) words — it grows until size m,
    then slides forward by dropping the oldest and adding the newest.
    """
    log_prob_sum = 0.0
    n = 0

    # Initialize cache counter with the first two words (indices 0 and 1)
    # These are the words before we start predicting (from index 2 onwards)
    cache_counter = Counter(tokens[:2])
    cache_size    = 2   # tracks how many words are currently in the cache

    for i in range(2, len(tokens)):
        w1, w2, w3 = tokens[i-2], tokens[i-1], tokens[i]

        # P_cache
        if cache_size > 0:
            p_c = cache_counter.get(w3, 0) / cache_size # counts how many times w3 appears in the cache_counter (which has the last m tokens)
        else:
            p_c = 0.0

        # SUm of probabilities
        p = (1 - lam) * p_jm(w1, w2, w3) + lam * p_c
        p = max(p, 1e-10)
        log_prob_sum += math.log(p)
        n += 1

        
        # When the size of the cache_counter (cache_size) is below m, we want the cache_counter to keep adding tokens (until we reach a counter_size=m)
        if cache_size < m:
            # Growing phase: just add the new word
            cache_counter[w3] += 1
            cache_size += 1
        
        # When we are in token m+1, we already have m tokens before; so now, for the counter, we will include the current token
        # under prediction in the counter, and delete the first in the counter list, since that is what P_cache needs for the next prediction
        else:
            oldest = tokens[i - m] 
            cache_counter[oldest] -= 1 # we remove one instance of the token that was the oldest (it falls out of the window)

            # If there are no more instances of the one that was the oldest in the new window, we delete the token it from the cache_counter
            if cache_counter[oldest] == 0:
                del cache_counter[oldest]
            cache_counter[w3] += 1
            # No change in cache_size, since it is already size m

    return math.exp(-log_prob_sum / n)

# grid search
lambda_options = np.arange(0.01, 0.5, 0.01)
m_options = [10, 25, 50, 100, 200, 250, 300, 500, 1000, 2000]


total   = len(lambda_options) * len(m_options)
print(f"Searching over {total} configurations \n")

grid_results = []

for m in m_options:
    print(f'm={m} under study')
    for lam in lambda_options:
        val_pp = compute_perplexity_cache(valid_tokens, lam, m)
        grid_results.append((m, lam, val_pp))
        #print(f"m={m:5d}, lambda={lam:.0e} | val_perplexity={val_pp:.2f}")

# best configuration
best_m, best_lam, best_val_pp = min(grid_results, key=lambda x: x[2])
print(f"\nBest config: m={best_m}, lambda={best_lam:.0e} | val_perplexity={best_val_pp:.2f}")

# test perplexity
test_pp = compute_perplexity_cache(test_tokens, best_lam, best_m)
print(f"Test perplexity: {test_pp:.2f}")

Searching over 490 configurations 

m=10 under study
m=25 under study
m=50 under study
m=100 under study
m=200 under study
m=250 under study
m=300 under study
m=500 under study
m=1000 under study
m=2000 under study

Best config: m=250, lambda=1e-01 | val_perplexity=160.66
Test perplexity: 153.78


# Exercise 5.5

Finally, we want to compare the generation ability of the last model (JM+cache) and of the LSTM. 

**JM+cache:**
For this matter, we first see which are the unigrams, bigrams and trigrams that appear the most in the training data. With these in mind, we can generate an initial prompt using some of these tokens combinations, so the model is evaluated in a context in which it is 'well trained'.

In [95]:
# Most frequent unigrams, bigrams and trigrams
print("Top 15 unigrams:")
for word, count in counts1.most_common(15):
    print(f"  {word:20s} {count}")

print("Top 15 bigrams:")
for bigram, count in counts2.most_common(15):
    print(f"  {str(bigram):35s} {count}")

print("Top 15 trigrams:")
for trigram, count in counts3.most_common(15):
    print(f"  {str(trigram):50s} {count}")

# Vocabulary (words in training set)
vocabulary = list(counts1.keys())

Top 15 unigrams:
  the                  50770
  <unk>                45020
  <eos>                42068
  N                    32481
  of                   24400
  to                   23638
  a                    21196
  in                   18000
  and                  17474
  's                   9784
  that                 8931
  for                  8927
  $                    7541
  is                   7337
  it                   6112
Top 15 bigrams:
  ('$', 'N')                          7475
  ('N', 'N')                          7183
  ('<eos>', 'the')                    7057
  ('of', 'the')                       5290
  ('in', 'the')                       4503
  ('N', 'million')                    4493
  ('<unk>', '<unk>')                  4010
  ('the', '<unk>')                    3867
  ('<unk>', '<eos>')                  3325
  ('N', '<eos>')                      2526
  ('a', '<unk>')                      2449
  ('N', 'to')                         2050
  ('<unk>', 'and')    

## Sampling tokens from probability

In order to generate in a new word, the argmax procedure was used at the beginning, i.e. we compute the probability $P(token|w_{i-2}, w_{i-1}, cache)$ for every possible token, and the token with the highest probability, is selected. Results, however, were not positive.

In this case, we will use another procedure, that, although still being simple, gives better results. Here, we sample over the probabilities to select the next token; i.e. for each possible token coming next, we compute $P(token|w_{i-2}, w_{i-1}, cache)$, and we select that token based on its probability; e.g. if $P('dog' |w_{i-2}, w_{i-1}, cache)=0.6$, and $P('cat' |w_{i-2}, w_{i-1}, cache)=0.4$, then if we have a context of $(w_{i-2}, w_{i-1}, cache)$, 60% of the times, 'dog' will be selected as the next token, and 40% of the times, 'cat' will be the chosen token. With this approach, more varied results appear.

For the sentence generation of the LSTM model, we will feed the tokens from the prompt to the LSTM, and after that, an iterative process will generate the next tokens in the sentence using the same probabilistic approach as for the n-gram model. Taking into account the previous tokens, the LSTM assigns a probability to each token in the vocabulary; since all of them are candidates for being the next token; and based on these probabilities, a token from the vocabulary is selected (next token in the sentence).

In [97]:

prompts = [
    "the company said",
    "earnings fell from the year-ago",
    "the lower net included a charge of",
    "sales amounted to $ N million",
    "the president said he would",
    'the $ N million that'
]


In [98]:
import random
from collections import Counter


def sample_next_token(scores: dict) -> str:
    "We could use argmax for the sentence generation, but it might be better to do this:"
    "Given the dictionary {word: probability}, we select a token proportional to its probability: if P(token_o|w_{i-2}, w_{i-1}, cache)=0.6,"
    "the token_o will be generated 60% of times given a context of (w_{i-2}, w_{i-1}, cache)."
    words = list(scores.keys()) # we extract the words from the dictionary
    probs = list(scores.values()) # we extract the probabilities for each of these words

    # We ensure that probabilities sum to 1, just in case
    total = sum(probs)
    probs = [p / total for p in probs]

    # Draw a random number in [0, 1). This will determine the probability
    r = random.random()
    cumulative = 0.0
    for word, p in zip(words, probs):
        cumulative += p
        if r <= cumulative:
            return word
    return words[-1] # if there is an error, just for the program not to present an error


def generate_sentence(prompt, lam, m, max_words=40, seed=None):
    "This function generates a sentence given a prompt using the model from step 4"

    if seed is not None:
        random.seed(seed)

    tokens = prompt.strip().split() # we turn the prompt into a list of words

    # We need at least 2 context tokens to form a trigram. So if we have less, we add 'the' (padding)
    while len(tokens) < 2:
        tokens = ['the'] + tokens

    # Using the prompt given, we intialize cache
    cache_start   = max(0, len(tokens) - m)
    cache_window  = list(tokens[cache_start:]) # cache values
    cache_counter = Counter(cache_window) # we count frequency of tokens in the cache window
    cache_size    = len(cache_window)

    for _ in range(max_words):
        w1 = tokens[-2] # penultimate token
        w2 = tokens[-1] # last token

        
        # We compute the probability for every word in the vocabulary usinng the probability shape of exercise 4
        scores = {}
        for w3 in vocabulary:
            p_base  = p_jm(w1, w2, w3)
            p_cache = cache_counter.get(w3, 0) / cache_size if cache_size > 0 else 0.0
            scores[w3] = (1.0 - lam) * p_base + lam * p_cache

        # Selecting next token using the function from before
        next_token = sample_next_token(scores)

        tokens.append(next_token) # we keep building the sentence

        # we stop if the model generates an end of sentence
        if next_token == '<eos>':
            break

        # We update the cache with the new token the same way as we did before
        if cache_size < m:
            cache_counter[next_token] += 1
            cache_size += 1
            cache_window.append(next_token)
        else:
            oldest = cache_window.pop(0)
            cache_counter[oldest] -= 1
            if cache_counter[oldest] == 0:
                del cache_counter[oldest]
            cache_counter[next_token] += 1
            cache_window.append(next_token)

    return ' '.join(tokens)



print(f"Generating with lambda={best_lam:.2f}, m={best_m}")

for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    for i in range(5):
        sentence = generate_sentence(prompt, lam=best_lam, m=best_m, seed=i)
        print(f"  [{i+1}] {sentence}")

Generating with lambda=0.10, m=250

Prompt: 'the company said'
  [1] the company said ross ross who make company 's net rose and so traders have said so ross and physical limitations smoke unclear its going reservations so he the average 30-day estimate extensive so salary of going on <eos>
  [2] the company said <eos>
  [3] the company said breeden avoided <eos>
  [4] the company said the way to do n't the <unk> region N the probability it seems to bid the best chance with best loss <unk> concept one to <unk> programs in addition chance issue opinion that failed to shift wo a the to
  [5] the company said the the by the <unk> that aim yesterday yesterday the same buildings the will to fulfill i lawmakers survey a at first thing especially weak <unk> systems but by the last a proving costly vice for vacation million educational experience

Prompt: 'earnings fell from the year-ago'
  [1] earnings fell from the year-ago quarter earnings were at from 's policy and had been transformed by 

In [99]:
# We build a model and get the best weights from the training. we now set batch_size and bppt length to 1.
inference_model = build_model(num_layers=2, lstm_size=650, batch_size=1, bptt_len=1, vocab_size=vocab_size)
inference_model.set_weights(model.get_weights())

def generate_sentence_lstm(prompt, model, tokenizer, max_words=40, seed=None):
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)
    
    tokens = prompt.strip().split() # we tokenize the prompt string into a word list
    reset_model_states(model)
    
    # We feed the model with the tokens from the prompt
    for token in tokens:
        token_id = tokenizer.word_index.get(token, 0) # we convert the word to integer id so that the model understands. 0 if it is unknown
        x = np.array([[token_id]])  # shape (1, 1)
        model(x, training=False)   # we update states, and discard output (no generation here). (batch, timestep)
    
    # Now generate
    last_token = tokens[-1] # we start generating from the last token given by the prompt
    for _ in range(max_words):
        token_id = tokenizer.word_index.get(last_token, 0)
        x = np.array([[token_id]])  # shape (1, 1)
        probs = model(x, training=False).numpy()[0, 0]  # length is vocab_size. These are the probabilities assigned to each token in the model
        probs = probs / probs.sum() # we normalize probabilities
        next_id = np.random.choice(len(probs), p=probs) # we choose a token according to its probability (instead of drawing a random number, this is cleaner)
        next_token = tokenizer.index_word.get(next_id, '<unk>') # we turn id into word. unk if id has not been found
        tokens.append(next_token)
        last_token = next_token # the token generated will be the input for the next token generation
        if next_token == '<eos>':
            break
    
    return ' '.join(tokens)

print("LSTM generation:")
for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    for i in range(5):
        sentence = generate_sentence_lstm(
            prompt,
            model=inference_model,
            tokenizer=tokenizer,
            seed=i
        )
        print(f"  [{i+1}] {sentence}")

LSTM generation:

Prompt: 'the company said'
  [1] the company said it may continue to report by a benefit side of american employees eos investor unk unk the talks with corporate freedom for a staff of japanese business provisions in n n from unk over the bid to help virtually go
  [2] the company said that other the time to unk the companies are being used for the owners eos some companies one of the partners involved in companies began buying the unk of family to buy valley national co of his machines eos even
  [3] the company said it 's third quarter earnings of n million or n cents a share in the state earlier period eos the two million three was unk an n n issue in a n million increase in n its net is yesterday
  [4] the company said it may have a main mind eos to the wall street journal it is trying to unk its first day in most of the past six years eos in fact commodore has headed some u s memories that are n't
  [5] the company said peter unk r industry chairman of daiwa the 

# Discussion

In this case, a noticeable difference in the sentence generation can be appreciated. Although it cannot be considered that the LSTM has reasonable sentence generation, the sentences that it generates seem noticeably more natural that those generated by the n-gram model.

However, the statement for the exercises suggests a test perplexity of around 135 or below for the JM+cache model, while I get a perplexity of around 155, which is noticeably above this value. With an n-gram model of test perplexity around 135, the sentences would probably sound more natural than now. Nevertheless, with 135 of test perplexity, the difference would probably still be noticeable. Given that the test perplexity obtained for the LSTM model is around 70-80, an n-gram model with perplexity of 135 will likely still generate texts noticeably more confusing that those created by the LSTM.

Although interpretability is completely lost when using the LSTM, the significant difference between perplexities might be enough reason for multiple users to choose the LSTM over the n-gram model.

That being said, it would be interesting to study modern black-box models and measure the performance that they have over this test set. 